In [ ]:
"""
Academic Paper Search LLM using Ollama and LangChain
Searches for research papers online and provides summaries with full titles and links
"""

from langchain_community.llms import Ollama
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.prompts import PromptTemplate
from langchain.tools import Tool
import re
import json

# Initialize Ollama LLM
llm = Ollama(
    model="llama3.2",  # Change to your preferred model
    temperature=0.3
)

def search_papers(query: str) -> str:
    """Search for academic papers and return structured results with titles and links"""
    # Add paper-specific search terms
    enhanced_query = f"{query} site:arxiv.org OR site:scholar.google.com OR site:pubmed.ncbi.nlm.nih.gov OR filetype:pdf research paper"
    
    search = DuckDuckGoSearchResults(num_results=10)
    results = search.run(enhanced_query)
    
    if not results or results == "No good DuckDuckGo Search Result was found":
        return "No papers found. Try refining your search query."
    
    # Parse results to extract titles and links
    formatted_results = parse_search_results(results)
    return formatted_results

def parse_search_results(results: str) -> str:
    """Parse search results to extract paper titles and links"""
    try:
        # Try to parse as list of dicts
        if results.startswith('['):
            results_list = eval(results)
            
            formatted = "Found the following papers:\n\n"
            for idx, item in enumerate(results_list, 1):
                title = item.get('title', 'No title')
                link = item.get('link', 'No link')
                snippet = item.get('snippet', '')
                
                formatted += f"{idx}. **{title}**\n"
                formatted += f"   Link: {link}\n"
                if snippet:
                    formatted += f"   Summary: {snippet}\n"
                formatted += "\n"
            
            return formatted
    except:
        pass
    
    # Fallback: try to extract from string format
    lines = results.split('\n')
    formatted = "Found the following papers:\n\n"
    count = 1
    
    for line in lines:
        # Look for title patterns
        title_match = re.search(r'title["\']?\s*[:=]\s*["\']([^"\']+)["\']', line, re.IGNORECASE)
        link_match = re.search(r'link["\']?\s*[:=]\s*["\']([^"\']+)["\']', line, re.IGNORECASE)
        
        if title_match and link_match:
            formatted += f"{count}. **{title_match.group(1)}**\n"
            formatted += f"   Link: {link_match.group(1)}\n\n"
            count += 1
    
    if count == 1:
        return results  # Return raw if parsing fails
    
    return formatted

# Create the paper search tool
paper_tool = Tool(
    name="PaperSearch",
    func=search_papers,
    description="Search for academic papers. Returns paper titles with direct links. Input should be a research topic."
)

tools = [paper_tool]

# Enhanced prompt to ensure links are included
template = """You are an academic research assistant specialized in finding research papers.

When a user asks about a research topic:
1. Use the PaperSearch tool to find relevant papers
2. Present the results exactly as provided by the tool, including ALL paper titles and links
3. Do NOT summarize or remove any links
4. Add brief context about the papers if relevant
5. Ensure every paper title is shown with its corresponding link

IMPORTANT: Always include the full paper titles and clickable links in your final answer.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer with ALL paper titles and links from the observations

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

def search_papers_query(query: str) -> str:
    """
    Main function to search for papers based on user query
    
    Args:
        query: User's research question or topic
        
    Returns:
        Formatted response with paper titles and links
    """
    try:
        response = agent_executor.invoke({"input": query})
        return response['output']
    except Exception as e:
        return f"Error processing query: {str(e)}"

# Main interface
if __name__ == "__main__":
    print("=" * 60)
    print("Academic Paper Search Assistant")
    print("Powered by Ollama + LangChain")
    print("=" * 60)
    print("\nSearches for research papers and provides full titles + links")
    print("Type 'quit' or 'exit' to stop\n")
    
    while True:
        try:
            user_query = input("\n🔍 Enter your research query: ").strip()
            
            if user_query.lower() in ['quit', 'exit', 'q']:
                print("\nGoodbye! Happy researching! 📚")
                break
            
            if not user_query:
                print("Please enter a valid query.")
                continue
            
            print("\n🤖 Searching for papers...\n")
            result = search_papers_query(user_query)
            print(f"\n📄 Results:\n{result}")
            print("\n" + "-" * 60)
            
        except KeyboardInterrupt:
            print("\n\nGoodbye! Happy researching! 📚")
            break
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            print("Please try again with a different query.")

C:\Users\ss348\AppData\Local\Temp\ipykernel_29916\2400573241.py:15: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(


Academic Paper Search Assistant
Powered by Ollama + LangChain

Searches for research papers and provides full titles + links
Type 'quit' or 'exit' to stop


🤖 Searching for papers...



> Entering new AgentExecutor chain...
Action: PaperSearch
Action Input: "Remote sensing in three years"snippet: Computer vision may enable sensitive detection of subtle movement patterns that escape the human eye, aligning with an emerging research focus on early disease stages. However, challenges in accessibility, ethics, and validation need to be addressed for widespread adoption., title: Computer Vision in Clinical Neurology: A Review - PubMed, link: https://pubmed.ncbi.nlm.nih.gov/39960732/, snippet: The protocol of a randomized trial is the foundation for study planning, conduct, reporting and external review. However, trial protocols vary in their completeness and often do not address key elements of design and conduct. The SPIRIT (Standard Protocol Items: Recommendations for Interventional Tr …,

In [ ]:
from system_api.search_engine import SearchEngine
search_engine = SearchEngine('./paper_entries.csv')
result = search_engine.search('neural network')

In [7]:
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set
from collections import defaultdict
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate


@dataclass
class ResearchNode:
    """Represents a single research paper in the inheritance tree"""
    year: int
    student: str
    title: str
    area: str
    trunk: str
    note: Optional[str] = None
    predecessor: Optional['ResearchNode'] = None
    successors: List['ResearchNode'] = field(default_factory=list)
    
    def extract_methods(self) -> Set[str]:
        """Extract methods/techniques from title"""
        title_lower = self.title.lower()
        methods = set()
        
        # Deep learning models
        model_keywords = [
            'cnn', 'lstm', 'rnn', 'gan', 'cgan', 'vae', 'autoencoder',
            'transformer', 'bert', 'gpt', 'llama', 'yolo', 'unet', 'resnet',
            'dnn', 'rbf', 'svm', 'random forest', 'xgboost',
            '卷積', '循環', '生成對抗', '神經網路', '類神經', '深度學習'
        ]
        
        # Techniques
        technique_keywords = [
            'ensemble', 'transfer learning', 'federated learning', 
            'attention', 'rag', 'fine-tuning', 'pre-training',
            'data augmentation', 'reinforcement learning',
            '集成', '遷移學習', '轉移學習', '聯邦學習', '強化學習',
            '注意力機制', '資料增強', '預訓練', '微調'
        ]
        
        # Special methods
        special_keywords = [
            'gaussian', 'mdl', 'stl', 'r-tree', 'space-mdl',
            '高斯', '時空', '分解', '樹狀結構'
        ]
        
        all_keywords = model_keywords + technique_keywords + special_keywords
        
        for keyword in all_keywords:
            if keyword in title_lower:
                methods.add(keyword)
        
        return methods
    
    def get_lineage_path(self) -> List[str]:
        """Get the full lineage path"""
        path = []
        current = self
        while current:
            path.append(f"{current.year} - {current.student}")
            current = current.predecessor
        return list(reversed(path))


class ResearchInheritanceAnalyzer:
    """
    Analyzes research inheritance patterns and generates natural language reports
    using LLM to explain how topics evolve and knowledge transfers between researchers.
    """
    
    def __init__(self, json_path: str, model: str = "llama3.2:latest", verbose: bool = True):
        """
        Initialize the analyzer
        
        Args:
            json_path: Path to relationships.json file
            model: Ollama model name
            verbose: Whether to print progress messages
        """
        self.verbose = verbose
        self._log("Initializing Research Inheritance Analyzer...")
        
        # Load data
        with open(json_path, 'r', encoding='utf-8') as f:
            self.forest_data = json.load(f)
        
        # Initialize LLM
        self._log(f"Loading LLM model: {model}...")
        self.llm = OllamaLLM(model=model, temperature=0.7)
        
        # Build tree
        self.nodes: Dict[str, ResearchNode] = {}
        self.nodes_by_name: Dict[str, ResearchNode] = {}
        
        self._build_inheritance_tree()
        self._log(f"✓ Loaded {len(self.nodes)} research nodes\n")
    
    def _log(self, message: str):
        if self.verbose:
            print(message)
    
    def _build_node(self, paper_data: Dict, area_name: str, trunk_name: str, 
                    predecessor: Optional[ResearchNode] = None) -> ResearchNode:
        """Build a ResearchNode from paper data"""
        node_id = f"{paper_data['year']}_{paper_data['student']}"
        
        if node_id in self.nodes:
            return self.nodes[node_id]
        
        node = ResearchNode(
            year=int(paper_data['year']),
            student=paper_data['student'],
            title=paper_data['title'],
            area=area_name,
            trunk=trunk_name,
            note=paper_data.get('note'),
            predecessor=predecessor
        )
        
        self.nodes[node_id] = node
        self.nodes_by_name[node.student] = node
        
        if predecessor:
            predecessor.successors.append(node)
        
        # Recursively build successors
        for child_data in paper_data.get('children', []):
            self._build_node(child_data, area_name, trunk_name, predecessor=node)
        
        return node
    
    def _build_inheritance_tree(self):
        """Build the complete inheritance tree"""
        for area in self.forest_data['research_forest']:
            area_name = area['area_name']
            
            for trunk in area['trunks']:
                trunk_name = trunk['trunk_name']
                
                for lineage_root in trunk['lineage']:
                    self._build_node(lineage_root, area_name, trunk_name)
    
    def _get_all_successors(self, node: ResearchNode) -> List[ResearchNode]:
        """Get all successors recursively"""
        successors = []
        for successor in node.successors:
            successors.append(successor)
            successors.extend(self._get_all_successors(successor))
        return successors
    
    def _find_researcher(self, name: str) -> Optional[ResearchNode]:
        """Find researcher by partial name match"""
        for node in self.nodes_by_name.values():
            if name.lower() in node.student.lower():
                return node
        return None
    def list_authors(
        self,
        area: Optional[str] = None,
        trunk: Optional[str] = None,
        sort: str = "alpha",          # "alpha" or "year"
        include_year: bool = False,   # if True, return ["2019 李東錡", ...]; else ["李東錡", ...]
    ) -> List[str]:
        """
        Return a unique list of available authors (students) to select from.

        Args:
            area: filter by area_name (exact match). Example: "交通數據分析"
            trunk: filter by trunk_name (exact match). Example: "人流分析與預測"
            sort: "alpha" (A→Z by name) or "year" (ascending by first appearance year)
            include_year: if True, include the earliest year next to name for display

        Returns:
            A list of author display strings.
        """
        # collect candidates
        candidates = []
        for node in self.nodes.values():
            if area and node.area != area:
                continue
            if trunk and node.trunk != trunk:
                continue
            candidates.append((node.student, node.year))

        # aggregate to earliest year per author
        first_year_by_author: Dict[str, int] = {}
        for name, yr in candidates:
            if name not in first_year_by_author or yr < first_year_by_author[name]:
                first_year_by_author[name] = yr

        # sorting
        if sort == "year":
            ordered = sorted(first_year_by_author.items(), key=lambda kv: (kv[1], kv[0]))
        else:  # alpha
            ordered = sorted(first_year_by_author.items(), key=lambda kv: kv[0])

        # formatting
        if include_year:
            return [f"{yr} {name}" for name, yr in ordered]
        else:
            return [name for name, _ in ordered]

    def analyze_research_lineage(self, pioneer_researcher: str) -> str:
        """
        Generate a report about a complete research lineage
        
        Args:
            pioneer_researcher: Name of the pioneering researcher
            
        Returns:
            LLM-generated lineage report
        """
        pioneer = self._find_researcher(pioneer_researcher)
        
        if not pioneer:
            return f"找不到研究者：{pioneer_researcher}"
        
        # Make sure it's a pioneer (no predecessor)
        if pioneer.predecessor is not None:
            pioneer = pioneer.predecessor
            while pioneer.predecessor is not None:
                pioneer = pioneer.predecessor
        
        # Build complete lineage
        def build_lineage_tree(node: ResearchNode, level: int = 0) -> Dict:
            methods = node.extract_methods()
            entry = {
                "level": level,
                "generation": f"第{level + 1}代" if level > 0 else "開創",
                "researcher": node.student,
                "year": node.year,
                "title": node.title,
                "methods": list(methods)
            }
            
            if node.predecessor:
                pred_methods = node.predecessor.extract_methods()
                entry["based_on"] = node.predecessor.student
                entry["inherited_methods"] = list(methods.intersection(pred_methods))
                entry["innovations"] = list(methods - pred_methods)
            
            if node.successors:
                entry["successors"] = [build_lineage_tree(s, level + 1) for s in node.successors]
            
            return entry
        
        lineage = build_lineage_tree(pioneer)
        all_successors = self._get_all_successors(pioneer)
        
        context = {
            "lineage": lineage,
            "total_generations": max([len(s.get_lineage_path()) for s in all_successors]) if all_successors else 1,
            "total_researchers": 1 + len(all_successors),
            "year_span": f"{pioneer.year}-{max(s.year for s in all_successors)}" if all_successors else str(pioneer.year),
            "area": pioneer.area,
            "trunk": pioneer.trunk
        }
        
        # Generate report
        prompt = PromptTemplate(
            input_variables=["context"],
            template="""你是一位學術傳承研究專家。請根據以下研究工作傳承數據，用中文撰寫一份詳細的傳承分析報告。

            傳承數據：
            {context}

            請撰寫一份完整的研究工作傳承報告，包含以下部分，都用列點或一句話說 (請講的超級簡潔，除了標題要完整列出以外)：

            1. 【誰繼承誰】

            2. 【年分與方法論演變】

            3. 【可能未來工作】

            """
        )
        
        context_str = json.dumps(context, ensure_ascii=False, indent=2)
        report = self.llm.invoke(prompt.format(context=context_str))
        
        return report
    

"""Demonstrate the ResearchInheritanceAnalyzer"""
print("\n" + "="*80)
print("RESEARCH INHERITANCE ANALYZER - LLM REPORT GENERATOR")
print("="*80 + "\n")

# Initialize analyzer (only needs JSON file)
analyzer = ResearchInheritanceAnalyzer(
    json_path='./json_files/relationships.json',
    verbose=True
)

# Example 1: Analyze single researcher
print("="*80)
print("範例 1: 研究工作傳承分析")
print("="*80)
report = analyzer.analyze_research_lineage('賴紫平')
print(report)
print("\n")
print(analyzer.list_authors())



RESEARCH INHERITANCE ANALYZER - LLM REPORT GENERATOR

Initializing Research Inheritance Analyzer...
Loading LLM model: llama3.2:latest...
✓ Loaded 29 research nodes

範例 1: 研究工作傳承分析
**研究工作傳承報告**

**1.【誰繼承誰】**

* 賴紫平（開創）->鄭佳昇（第2代）

**2.【年分與方法論演變】**

* 2020:賴紫平的研究《使用深度學習模型結合二維高斯函數探討交通站點之用量與成因》為基礎，引入了深度學習和高斯函數的方法論。
* 2022:鄭佳昇（第2代）延續了賴紫平的研究精神，並提出了一個新的研究題目《基於3D-RCL與條件式生成對抗網路產生未來時刻人群分布之可能性探討》，並引入了生成對抗方法論。

**3.【可能未來工作】**

* 基於前置的研究成果，下一步可能會考慮以下幾個方向：
 + 廣泛應用深度學習和高斯函數等方法於其他交通數據分析領域。
 + further 研究生成對抗方法論在人群分布預測中的有效性。
 + exploring new methodological frameworks 和 techniques來增進人流分析與預測的精確性。


['吳文成', '吳昺儒', '周映均', '施長宏', '李東錡', '李軍磊', '李靜瑜', '李香蘭', '林佑蓉', '林子雁', '林聖晧', '楊添翼', '楊竣傑', '歐晉佑', '江吉晟', '洪意雯', '王怡婷', '王詠緹', '盧政傑', '許畯棠', '賴紫平', '邱聖崴', '鄭佳昇', '鄭力誠', '鍾久祿', '陳姿靜', '陳柏翔', '陳詳元', '馬承安']
